In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist #mnist 훈련셋과 테스트셋
from tensorflow.keras.utils import to_categorical #원핫인코딩
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from matplotlib import pyplot as plt #학습과정 loss와 acc시각화
# quiz에서는 scale조정, train_test_split 등을 추가

- Red Wine 등급 계층
1. 데이터 셋 확보 및 전처리
    csv -> 결측치 처리 -> 독립변수와 타겟변수 분리 -> 독립변수 스케일조정,
    -> 타겟변수의 원핫인코딩 -> 훈련셋과 테스트셋분리(train_test_split이용 층화추출)
2. 모델 구성(입력11, 출력6-pd.getdummies | 출력9-to_categorical) layer층 4
3. 모델 학습 과정 설정
4. 모델 학습(callbacks이용)
5. 모델 평가(그래프, 평가, 교차표)
6. 모델저장&사용

# 1. 데이터 확보 & 전처리

In [13]:
#데이터 읽어오기
# fixed acidity : 고정 산도
# volatile acidity : 휘발성 산도
# citric acid : 시트르산
# residual sugar : 잔류 당분
# chlorides : 염화물
# free sulfur dioxide : 자유 이산화황
# total sulfur dioxide : 총 이산화황
# density : 밀도
# pH
# sulphates : 황산염
# alcohol
# quality : 0 ~ 10(높을 수록 좋은 품질)
redwine = pd.read_csv('data/winequality-red.csv', sep=';')
redwine['quality'].value_counts()

5    681
6    638
7    199
4     53
8     18
3     10
Name: quality, dtype: int64

In [14]:
redwine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [15]:
#결측치 처리(중앙값)
redwine.fillna(value=redwine.median(), inplace=True)

In [16]:
redwine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


In [28]:
#독립변수와 타겟변수 분리
X_train = redwine.iloc[:1300, :-1].values
y_train = redwine.iloc[:1300,-1].values
X_val = redwine.iloc[1300:1400, :-1].values
y_val = redwine.iloc[1300:1400, -1].values
X_test = redwine.iloc[1400:, :-1].values
y_test = redwine.iloc[1400:, -1].values

In [29]:
#독립변수 스케일조정(StandardScaler)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_X = scaler.fit_transform(X_train)
scaled_y = scaler.fit_transform(y_train.reshape(-1,1))

In [32]:
#타겟변수의 원핫인코딩
from tensorflow.keras.utils import to_categorical
train_Y = to_categorical(y_train)
val_Y = to_categorical(y_val)
test_Y = to_categorical(y_test)
train_Y.shape, val_Y.shape, test_Y.shape

((1300, 9), (100, 8), (199, 9))

In [35]:
#훈련셋과 테스트셋분리(train_test_split이용 층화추출)
from sklearn.model_selection import train_test_split
X_train, y_train, X_test,y_test = train_test_split(scaled_X, scaled_y,
                                                  test_size=0.3,
                                                  random_state=7)

# 2. 모델 구성

In [41]:
#모델 생성
model = Sequential()
model.add(Input(shape=(11,)))
model.add(Dense(units=2, activation='relu'))
model.add(Dense(units=7, activation='relu'))
model.add(Dense(units=10, activation='relu'))
model.add(Dense(units=9, activation='sigmoid'))
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_8 (Dense)             (None, 2)                 24        
                                                                 
 dense_9 (Dense)             (None, 7)                 21        
                                                                 
 dense_10 (Dense)            (None, 10)                80        
                                                                 
 dense_11 (Dense)            (None, 9)                 99        
                                                                 
Total params: 224
Trainable params: 224
Non-trainable params: 0
_________________________________________________________________


# 3. 모델 학습 과정 설정